# Compare cl 
This document checks if every model has been 

In [1]:
! rm *.csv

from google.colab import files
uploaded = files.upload()
%ls

Saving 2h_price.csv to 2h_price.csv
Saving 2h_sentiment.csv to 2h_sentiment.csv
2h_price.csv  2h_sentiment.csv  sample_data/


## Setup

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
import random

In [4]:
from os import listdir
from os.path import isfile

Importing the collected data

In [5]:
cryptos = ["biance-coin", "bitcoin-cash", "bitcoin", "cardano", "chainlink", "ethereum", "litecoin", "ripple", "stellar"]

## Helper Methods

In [6]:
def plot_time_series(predicted, true, n_training, filename):
  """
  Plot the time series
  """
  plt.figure(figsize=(8,6)) #plotting
  plt.axvline(x=n_training, color="#ffd166", linestyle='-') #size of the training set

  plt.plot(predicted, label='Predicted Price', color="#118ab2") #predicted plot
  plt.plot(true, label='True Price', color="#06d6a0") #actual plot

  plt.title('Time-Series Prediction', fontsize=16)
  plt.xlabel('Time', fontsize=14)
  plt.ylabel('Price', fontsize=14)

  plt.xlim(0)
  plt.legend()
  plt.show()
  #plt.savefig(filename) 

In [7]:
def classify(predicted_best, true_best):
    df = pd.DataFrame([predicted_best, true_best], columns=["pred","true"])

    total_correct = (predicted_best == true_best).sum()
    return totalcorrect/len(predict)

## LSTM Model


### Model Definition

LSTM Class used: https://pytorch.org/docs/master/generated/torch.nn.LSTM.html#torch.nn.LSTM

Some tutorials: https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html



In [8]:
class LSTMCustom(nn.Module):
    def __init__(self, num_classes, input_size, hidden_size, num_layers, seq_length):
        super(LSTMCustom, self).__init__()
        self.num_classes = num_classes #number of classes
        self.num_layers = num_layers #number of layers
        self.input_size = input_size #input size
        self.hidden_size = hidden_size #hidden state
        self.seq_length = seq_length #sequence length

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout = 0.05
            )
        #self.fc_1 =  nn.Linear(hidden_size, 128) #fully connected 1
        #self.fc = nn.Linear(128, num_classes) #fully connected last layer

        #self.relu = nn.ReLU()

        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self,x):
        h_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #hidden state
        c_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #internal state
        # Propagate input through LSTM
        output, (hn, cn) = self.lstm(x, (h_0, c_0)) #lstm with input, hidden, and internal state

        out = self.fc(output[:,-1,:])
        return out

### Model Parameters

In [9]:
num_epochs = 100
learning_rate = 0.01 #0.001 lr

hidden_size = 32 #number of features in hidden state
num_layers = 8 #number of stacked lstm layers

num_classes = len(cryptos)

look_back = 12

In [10]:
criterion = torch.nn.MSELoss()  # mean-squared error for regression

In [11]:
mm = MinMaxScaler()
ss = StandardScaler()

le = LabelEncoder()

### Methods

In [12]:
def train_test_split_tensor(x, y):
  """
  Custom train/test splitting
  TODO: maybe shorten code by using sklearn.preprocessing.train_test_split
  """
  cutoff = round(x.shape[0] * 0.8)

  # split into train and test
  x_train = x[:cutoff, :]
  x_test = x[cutoff:,:]
  y_train = y[:cutoff, :]
  y_test = y[cutoff:, :]
  
  return x_train, x_test, y_train, y_test, cutoff

In [13]:
def convert_2d(x_ss):
  x_2d = []
  for index in range(look_back, len(x_ss)):
    x_2d.append(np.array(x_ss[index-look_back:index+1]))
  x_2d = np.array(x_2d)
  return x_2d

In [14]:
def get_x_y(df):
  # split into x and y
  y = le.fit_transform(df["best_crypto"])
  y = nn.functional.one_hot(torch.tensor(y, dtype=torch.int64), len(cryptos))

  x = df.drop(["best_crypto", "best_diff"],axis=1)
  # x_ss = ss.fit_transform(x)
  x_ss = x.values

  x_2d = torch.tensor(x_ss)
  if True:
    x_2d = convert_2d(x_ss)
    x_2d = Variable(torch.Tensor(x_2d))
    y = y[:len(y)-look_back]
  else:
    x_2d = Variable(torch.reshape(x_2d, (x_2d.shape[0], 1, x_2d.shape[1])))
  
  return x_2d, y

In [15]:
def train(x_train, y_train, lstm):
  """
  Train the lstm
  """

  for epoch in range(num_epochs):
    outputs = lstm.forward(x_train) #forward pass
    optimizer.zero_grad() #caluclate the gradient, manually setting to 0
  
    # obtain the loss function
    loss = criterion(outputs, y_train)
  
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
      print("Epoch: %d, loss: %1.5f" % (epoch, loss.item())) 

  return lstm, cutoff

## Execution

### Price and Sentiment

In [16]:
df = pd.read_csv("2h_sentiment.csv")

x_2d, y = get_x_y(df)
x_train, x_test, y_train, y_test, cutoff = train_test_split_tensor(x_2d, y)


In [17]:
print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)


torch.Size([3494, 13, 675]) torch.Size([874, 13, 675])
torch.Size([3494, 9]) torch.Size([874, 9])


In [18]:
input_size = x_train.size(2)
input_size

675

In [19]:
lstm = LSTMCustom(num_classes, input_size, hidden_size, num_layers, df.shape[1]) 
optimizer = torch.optim.Adam(lstm.parameters(), lr=learning_rate) 

lstm, cutoff = train(x_train.float(), y_train.float(), lstm)

Epoch: 0, loss: 0.13196
Epoch: 50, loss: 0.07446


#### Results

In [22]:
preds = lstm(x_2d.float()).argmax(axis=1)
y_true = y.argmax(axis=1)

In [23]:
train_pred = preds[:cutoff]
train_y = y_true[:cutoff]

train_acc = (train_pred == train_y).sum()/len(preds)
train_acc

tensor(0.4366)

In [24]:
test_pred = preds[cutoff:]
test_y = y_true[cutoff:]

test_acc = (test_pred == test_y).sum()/len(preds)
test_acc

tensor(0.1291)

In [25]:
print("input size: ", input_size)
print("epochs: ", num_epochs)
print("learning rate: ", learning_rate)
print("hidden size: ", hidden_size)
print("look back: ", look_back)

input size:  675
epochs:  100
learning rate:  0.01
hidden size:  32
look back:  12


In [26]:
pd.DataFrame([[train_acc, test_acc]], columns=["train accuracy", "test accuracy"])

,train accuracy,test accuracy
0,tensor(0.4366),tensor(0.1291)


### Price Only


In [27]:
df = pd.read_csv("2h_price.csv")

x_2d, y = get_x_y(df)
x_train, x_test, y_train, y_test, cutoff = train_test_split_tensor(x_2d, y)


In [28]:
print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)


torch.Size([3502, 13, 27]) torch.Size([876, 13, 27])
torch.Size([3502, 9]) torch.Size([876, 9])


In [29]:
input_size = x_train.size(2)
input_size

27

In [30]:
lstm = LSTMCustom(num_classes, input_size, hidden_size, num_layers, df.shape[1]) 
optimizer = torch.optim.Adam(lstm.parameters(), lr=learning_rate) 

lstm, cutoff = train(x_train.float(), y_train.float(), lstm)

Epoch: 0, loss: 0.10397
Epoch: 50, loss: 0.07442


#### Results


In [31]:
preds = lstm(x_2d.float()).argmax(axis=1)
y_true = y.argmax(axis=1)

In [32]:
train_pred = preds[:cutoff]
train_y = y_true[:cutoff]

train_acc = (train_pred == train_y).sum()/len(preds)
train_acc

tensor(0.4367)

In [33]:
test_pred = preds[cutoff:]
test_y = y_true[cutoff:]

test_acc = (test_pred == test_y).sum()/len(preds)
test_acc

tensor(0.1291)

In [34]:
print("input size: ", input_size)
print("epochs: ", num_epochs)
print("learning rate: ", learning_rate)
print("hidden size: ", hidden_size)
print("look back: ", look_back)

input size:  27
epochs:  100
learning rate:  0.01
hidden size:  32
look back:  12


In [35]:
pd.DataFrame([[train_acc, test_acc]], columns=["train accuracy", "test accuracy"])

,train accuracy,test accuracy
0,tensor(0.4367),tensor(0.1291)


### Save Model

In [36]:
# torch.save(lstm.state_dict(), "predict_crypto.model")
# files.download("predict_crypto.model")